# DS5 Step 2 Path F — Hierarchical Two-Stage Phenotyping

**Notebook:** `103_ds5_hierarchical_phenotype.ipynb`  
**Stage:** Stage 4, Step 2 — Path F (third of six paths)  
**Date:** 2026-04-19

## Purpose

Construct a two-level phenotype taxonomy combining:
- **Stage 1:** Phase 1 network-structural split (Profile A vs Profile B) — no new clustering
- **Stage 2:** Behavior-only sub-clustering within each profile on `[sig_hat, H_emp, x_mean]`

**Key motivation:** Path C (notebook 101) confirmed Phase 1 is entirely NeT-driven (ARI vs.
NeT-only = 0.990). Path A (notebook 102) found a stable behavior-only phenotype structure
(k=3, Jaccard=0.991) that is almost perfectly orthogonal to Phase 1 (ARI = 0.020). The
near-zero ARI means network type and behavioral type are two independent dimensions — no
single flat clustering captures both. Path F crosses them hierarchically.

**Step 2 only.** No Step 3 personalization, no intervention prescriptions, no causal claims.
All cluster labels are exploratory, DS5-specific, and require replication.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.metrics.cluster import calinski_harabasz_score, davies_bouldin_score
from scipy.stats import kruskal, chi2_contingency
import warnings
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT        = Path('..')
PHENOTYPING = ROOT / 'outputs' / 'model_outputs' / 'phenotyping'
FIGURES     = PHENOTYPING / 'figures'
FEATURES    = ROOT / 'outputs' / 'features' / 'ds5'
FIGURES.mkdir(parents=True, exist_ok=True)

# ── Constants ─────────────────────────────────────────────────────────────────
RANDOM_STATE  = 42
N_BOOTSTRAPS  = 100
SAMPLE_FRAC   = 0.80
MIN_CLUSTER   = 30
CORE_FEATURES = ['sig_hat', 'H_emp', 'x_mean']
PSYCH_SCALES  = ['fomo_mean_pre', 'belong_mean_pre', 'ostracism_mean_pre',
                 'digital_wellbeing_pre', 'social_media_wellbeing_pre']

print("Setup complete — Python/library versions verified")
print(f"  PHENOTYPING: {PHENOTYPING}")
print(f"  FIGURES:     {FIGURES}")


## 1. Load and prepare data

All four prior outputs are loaded and merged into a single master dataframe (N=414).
Phase 1 `profile_p1` is the Stage 1 split; Path C and Path A labels are post-hoc overlays.


In [ ]:
# Feature matrix (N=414, StoP-eligible)
fm = pd.read_csv(PHENOTYPING / 'ds5_phenotype_feature_matrix.csv')

# Phase 1 assignments — Stage 1 split
p1 = pd.read_csv(PHENOTYPING / 'ds5_cluster_assignments.csv')

# Path C sub-cluster assignments — post-hoc comparison
pc = pd.read_csv(PHENOTYPING / 'ds5_net_subcluster_assignments.csv')

# Path A behavior-only assignments — post-hoc comparison
pa = pd.read_csv(PHENOTYPING / 'ds5_behav_only_cluster_assignments.csv')

# Per-participant psychosocial survey scales
survey = pd.read_csv(FEATURES / 'ds5_survey_features.csv')

print(f"feature_matrix: {fm.shape}")
print(f"phase_1:        {p1.shape}")
print(f"path_c:         {pc.shape}")
print(f"path_a:         {pa.shape}")
print(f"survey:         {survey.shape}")


In [ ]:
# ── Master dataframe ──────────────────────────────────────────────────────────
KEEP_FEATS = CORE_FEATURES + ['H_hour', 'mu_hat', 'sigma_mu_ratio',
                               'HHI_net', 'rho_self', 'H_trans', 'H_app', 'H_out_resid']

df = fm[['participant_id_canonical', 'study_condition'] + KEEP_FEATS].copy()

# Stage 1 split
df = df.merge(p1[['participant_id_canonical', 'profile_p1']],
              on='participant_id_canonical', how='left')

# Path C combined sub-profile label
pc2 = pc[['participant_id_canonical', 'subprofile_B', 'subprofile_A']].copy()
pc2['subprofile_C'] = pc2['subprofile_B'].fillna('') + pc2['subprofile_A'].fillna('')
pc2['subprofile_C'] = pc2['subprofile_C'].replace('', np.nan)
df = df.merge(pc2[['participant_id_canonical', 'subprofile_C']],
              on='participant_id_canonical', how='left')

# Path A cluster labels
df = df.merge(pa[['participant_id_canonical', 'cluster_behav']],
              on='participant_id_canonical', how='left')

# Psychosocial scales
df = df.merge(survey[['participant_id_canonical'] + PSYCH_SCALES],
              on='participant_id_canonical', how='left')

print(f"Master df: {df.shape}")
print()
print("Stage 1 profile counts:")
print(df['profile_p1'].value_counts())
print()
print("Path A cluster_behav distribution:")
print(df['cluster_behav'].value_counts().sort_index())
print()
print("Path C subprofile_C distribution:")
print(df['subprofile_C'].value_counts())


In [ ]:
# Stage 2 sub-populations
B_df = df[df['profile_p1'] == 'Profile B'].copy().reset_index(drop=True)
A_df = df[df['profile_p1'] == 'Profile A'].copy().reset_index(drop=True)

print(f"Profile B (Stage 2 eligible): N = {len(B_df)}")
print(f"Profile A (Stage 2 eligible): N = {len(A_df)}")
print()
print("Profile B — Path A cluster distribution:")
print(B_df['cluster_behav'].value_counts().sort_index())
print()
print("Profile A — Path A cluster distribution:")
print(A_df['cluster_behav'].value_counts().sort_index())


## 2. Descriptive inspection within each profile

Inspect the behavioral feature space within Profile B and Profile A separately before
any clustering. This determines whether Stage 2 is operating in a well-structured
feature space or a near-uniform one.


In [ ]:
def describe_profile(name, sub):
    print("=" * 60)
    print(f"{name}  (N={len(sub)})")
    print("=" * 60)
    for feat in CORE_FEATURES + ['H_hour']:
        v = sub[feat]
        print(f"  {feat:20s}  mean={v.mean():.4f}  std={v.std():.4f}  "
              f"median={v.median():.4f}  IQR={v.quantile(0.75)-v.quantile(0.25):.4f}")
    print()
    print("  Correlation matrix (core features + H_hour):")
    print(sub[CORE_FEATURES + ['H_hour']].corr().round(3).to_string())
    print()

describe_profile("PROFILE B", B_df)
describe_profile("PROFILE A", A_df)


In [ ]:
pca_B = PCA(n_components=3, random_state=RANDOM_STATE)
pca_B.fit(StandardScaler().fit_transform(B_df[CORE_FEATURES].values))

pca_A = PCA(n_components=3, random_state=RANDOM_STATE)
pca_A.fit(StandardScaler().fit_transform(A_df[CORE_FEATURES].values))

for name, pca in [("Profile B", pca_B), ("Profile A", pca_A)]:
    ev = pca.explained_variance_ratio_
    print(f"{name} PCA (on {CORE_FEATURES}):")
    for i, v in enumerate(ev):
        print(f"  PC{i+1}: {v:.3f}  ({100*v:.1f}%)")
    print(f"  Cumulative PC1+2: {ev[:2].sum():.3f}")
    print()


In [ ]:
PA_COLORS  = {'C1': '#4575b4', 'C2': '#d73027', 'C3': '#91cf60'}
PA_MARKERS = {'C1': 'o',       'C2': 's',       'C3': '^'}

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.suptitle('Profile B — Behavioral feature space (N=289)\n'
             'Colored by Path A full-sample cluster', fontsize=12, fontweight='bold')

pairs = [('sig_hat', 'H_emp'), ('sig_hat', 'x_mean'), ('H_emp', 'x_mean')]
for ax, (xf, yf) in zip(axes, pairs):
    for cl in ['C1', 'C2', 'C3']:
        sub = B_df[B_df['cluster_behav'] == cl]
        ax.scatter(sub[xf], sub[yf],
                   c=PA_COLORS[cl], marker=PA_MARKERS[cl],
                   alpha=0.55, s=28, label=cl, edgecolors='none')
    ax.set_xlabel(xf, fontsize=10)
    ax.set_ylabel(yf, fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)

handles = [mpatches.Patch(color=PA_COLORS[c], label=c) for c in ['C1', 'C2', 'C3']]
axes[0].legend(handles=handles, title='Path A cluster', fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES / 'fig_f2_profileB_behav_space.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_f2_profileB_behav_space.png")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.suptitle('Profile A — Behavioral feature space (N=125)\n'
             'Colored by Path A full-sample cluster', fontsize=12, fontweight='bold')

for ax, (xf, yf) in zip(axes, pairs):
    for cl in ['C1', 'C2', 'C3']:
        sub = A_df[A_df['cluster_behav'] == cl]
        ax.scatter(sub[xf], sub[yf],
                   c=PA_COLORS[cl], marker=PA_MARKERS[cl],
                   alpha=0.55, s=28, label=cl, edgecolors='none')
    ax.set_xlabel(xf, fontsize=10)
    ax.set_ylabel(yf, fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)

handles = [mpatches.Patch(color=PA_COLORS[c], label=c) for c in ['C1', 'C2', 'C3']]
axes[0].legend(handles=handles, title='Path A cluster', fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES / 'fig_f3_profileA_behav_space.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_f3_profileA_behav_space.png")


## 3. Feature scaling

**Primary:** within-group `StandardScaler` — fit separately on Profile B and Profile A.
This is the principled choice for Stage 2: each group's behavioral variation is standardized
relative to that group's own distribution, not contaminated by between-profile differences.

**Sensitivity:** Full-sample scaler (same as Path A) — used only for the ARI sensitivity
check in Section 8.


In [ ]:
# Within-group scalers (primary)
scaler_B = StandardScaler()
X_B = scaler_B.fit_transform(B_df[CORE_FEATURES].values)

scaler_A = StandardScaler()
X_A = scaler_A.fit_transform(A_df[CORE_FEATURES].values)

# Full-sample scaler (sensitivity only)
scaler_full = StandardScaler()
scaler_full.fit(df[CORE_FEATURES].values)
X_B_full = scaler_full.transform(B_df[CORE_FEATURES].values)
X_A_full = scaler_full.transform(A_df[CORE_FEATURES].values)

print("Within-group scaler B — fitted on N=" + str(len(B_df)))
print("  means:", {f: round(float(m), 4) for f, m in zip(CORE_FEATURES, scaler_B.mean_)})
print()
print("Within-group scaler A — fitted on N=" + str(len(A_df)))
print("  means:", {f: round(float(m), 4) for f, m in zip(CORE_FEATURES, scaler_A.mean_)})


## 4. Stage 2 cluster selection within Profile B (N=289)

K-range: [2, 3, 4]. Minimum cell size enforced at N≥30. Selected k = highest silhouette
among valid k. Bootstrap Jaccard (100 bootstraps, 80% sample) required for selected k.


In [ ]:
def bootstrap_jaccard_stability(X, k, n_bootstraps=100, sample_frac=0.8, random_state=42):
    """Pairwise co-membership Jaccard across 100 bootstrap samples."""
    rng = np.random.RandomState(random_state)
    n = len(X)
    sample_n = int(n * sample_frac)
    km_full = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=random_state)
    labels_full = km_full.fit_predict(X)
    scores = []
    for _ in range(n_bootstraps):
        idx = rng.choice(n, size=sample_n, replace=False)
        X_boot = X[idx]
        km_boot = KMeans(n_clusters=k, init='k-means++', n_init=10,
                         random_state=int(rng.randint(10000)))
        labels_boot = km_boot.fit_predict(X_boot)
        lf = labels_full[idx]; lb = labels_boot
        ti, tj = np.tril_indices(sample_n, k=-1)
        sf = lf[ti] == lf[tj]; sb = lb[ti] == lb[tj]
        tp = (sf & sb).sum(); fp = (~sf & sb).sum(); fn = (sf & ~sb).sum()
        d = tp + fp + fn
        scores.append(tp / d if d > 0 else 0.0)
    return float(np.mean(scores)), float(np.std(scores))

print("Bootstrap Jaccard function defined")


In [ ]:
K_RANGE_B = [2, 3, 4]
b_results = []

print("Profile B k-selection (N=289)")
print("-" * 75)
for k in K_RANGE_B:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    lbs = km.fit_predict(X_B)
    sizes = pd.Series(lbs).value_counts()
    min_size = int(sizes.min())
    valid = min_size >= MIN_CLUSTER
    if valid:
        sil = silhouette_score(X_B, lbs)
        ch  = calinski_harabasz_score(X_B, lbs)
        db  = davies_bouldin_score(X_B, lbs)
    else:
        sil = ch = db = None
    b_results.append({'k': k, 'silhouette': sil, 'CH': ch, 'DB': db,
                      'min_cell': min_size, 'valid': valid})
    status = "VALID" if valid else ("REJECTED (min cell N=" + str(min_size) + " < " + str(MIN_CLUSTER) + ")")
    sil_s = f"{sil:.4f}" if sil is not None else "N/A"
    ch_s  = f"{ch:.1f}"  if ch  is not None else "N/A"
    db_s  = f"{db:.4f}"  if db  is not None else "N/A"
    print(f"  k={k}: sil={sil_s}  CH={ch_s}  DB={db_s}  min_cell={min_size}  {status}")

valid_b = [r for r in b_results if r['valid']]
SELECTED_KB = max(valid_b, key=lambda r: r['silhouette'])['k']
print()
print(f"Selected k for Profile B: {SELECTED_KB}")


In [ ]:
# Bootstrap Jaccard for selected k (and k+1 if also valid)
jaccard_B = {}
for r in b_results:
    if r['valid'] and r['k'] in [SELECTED_KB, SELECTED_KB + 1]:
        jmean, jstd = bootstrap_jaccard_stability(X_B, r['k'], N_BOOTSTRAPS, SAMPLE_FRAC, RANDOM_STATE)
        label = "STABLE" if jmean >= 0.75 else ("MARGINAL" if jmean >= 0.60 else "UNSTABLE")
        jaccard_B[r['k']] = {'mean': jmean, 'std': jstd, 'label': label}
        print(f"Profile B k={r['k']}: Jaccard={jmean:.3f} ± {jstd:.3f}  [{label}]")

JACCARD_KB_MEAN = jaccard_B[SELECTED_KB]['mean']
JACCARD_KB_STD  = jaccard_B[SELECTED_KB]['std']
JACCARD_KB_LABEL = jaccard_B[SELECTED_KB]['label']
print()
print(f"Primary stability (k={SELECTED_KB}): {JACCARD_KB_MEAN:.3f} ± {JACCARD_KB_STD:.3f}  [{JACCARD_KB_LABEL}]")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
ks     = [r['k'] for r in b_results]
sils   = [r['silhouette'] if r['silhouette'] is not None else np.nan for r in b_results]
chs    = [r['CH'] if r['CH'] is not None else np.nan for r in b_results]
dbs    = [r['DB'] if r['DB'] is not None else np.nan for r in b_results]

colors = ['#d73027' if k == SELECTED_KB else '#74add1' for k in ks]

for ax, vals, ylabel in zip(axes,
                              [sils, chs, dbs],
                              ['Silhouette', 'Calinski-Harabasz', 'Davies-Bouldin']):
    ax.bar(ks, vals, color=colors, edgecolor='white', linewidth=0.8)
    ax.axvline(SELECTED_KB, color='#d73027', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xlabel('k', fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_xticks(ks)
    ax.spines[['top', 'right']].set_visible(False)

_title_kb = str(SELECTED_KB)
_title_jac = str(round(JACCARD_KB_MEAN, 3))
fig.suptitle('Profile B Stage 2 k-selection  (selected k=' + _title_kb +
             ', Jaccard=' + _title_jac + ')', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'fig_f4_profileB_k_selection.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_f4_profileB_k_selection.png")


## 5. Stage 2 cluster selection within Profile A (N=125)

K-range: [2, 3] only. k=4 not attempted (N=125 too small). Min cell size N≥30 enforced.
If Jaccard < 0.75 for k=2, Profile A is reported descriptively (A_all) without sub-split.


In [ ]:
K_RANGE_A = [2, 3]
a_results = []

print("Profile A k-selection (N=125)")
print("-" * 75)
for k in K_RANGE_A:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    lbs = km.fit_predict(X_A)
    sizes = pd.Series(lbs).value_counts()
    min_size = int(sizes.min())
    valid = min_size >= MIN_CLUSTER
    if valid:
        sil = silhouette_score(X_A, lbs)
        ch  = calinski_harabasz_score(X_A, lbs)
        db  = davies_bouldin_score(X_A, lbs)
    else:
        sil = ch = db = None
    a_results.append({'k': k, 'silhouette': sil, 'CH': ch, 'DB': db,
                      'min_cell': min_size, 'valid': valid})
    status = "VALID" if valid else ("REJECTED (min cell N=" + str(min_size) + " < " + str(MIN_CLUSTER) + ")")
    sil_s = f"{sil:.4f}" if sil is not None else "N/A"
    ch_s  = f"{ch:.1f}"  if ch  is not None else "N/A"
    db_s  = f"{db:.4f}"  if db  is not None else "N/A"
    print(f"  k={k}: sil={sil_s}  CH={ch_s}  DB={db_s}  min_cell={min_size}  {status}")

valid_a = [r for r in a_results if r['valid']]
if valid_a:
    SELECTED_KA = max(valid_a, key=lambda r: r['silhouette'])['k']
    print()
    print(f"Selected k for Profile A: {SELECTED_KA}")
else:
    SELECTED_KA = None
    print("WARNING: No valid k found for Profile A — will report descriptively (A_all)")


In [ ]:
if SELECTED_KA is not None:
    jmean_a, jstd_a = bootstrap_jaccard_stability(X_A, SELECTED_KA, N_BOOTSTRAPS, SAMPLE_FRAC, RANDOM_STATE)
    jlabel_a = "STABLE" if jmean_a >= 0.75 else ("MARGINAL" if jmean_a >= 0.60 else "UNSTABLE")
    if jmean_a < 0.75:
        print(f"WARNING: Profile A k={SELECTED_KA} Jaccard={jmean_a:.3f} < 0.75 — UNSTABLE")
        print("  Profile A will be reported descriptively (A_all) without forced sub-split")
        PROFILE_A_SPLIT = False
    else:
        PROFILE_A_SPLIT = True
    JACCARD_KA_MEAN  = jmean_a
    JACCARD_KA_STD   = jstd_a
    JACCARD_KA_LABEL = jlabel_a
    print(f"Profile A k={SELECTED_KA}: Jaccard={jmean_a:.3f} ± {jstd_a:.3f}  [{jlabel_a}]")
else:
    PROFILE_A_SPLIT = False
    JACCARD_KA_MEAN  = None
    JACCARD_KA_STD   = None
    JACCARD_KA_LABEL = "N/A"
    print("Profile A: no valid k — descriptive only")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 4))
ks_a   = [r['k'] for r in a_results]
sils_a = [r['silhouette'] if r['silhouette'] is not None else np.nan for r in a_results]
chs_a  = [r['CH'] if r['CH'] is not None else np.nan for r in a_results]
dbs_a  = [r['DB'] if r['DB'] is not None else np.nan for r in a_results]

_sel_a = SELECTED_KA if SELECTED_KA is not None else -1
colors_a = ['#d73027' if k == _sel_a else '#74add1' for k in ks_a]

for ax, vals, ylabel in zip(axes,
                              [sils_a, chs_a, dbs_a],
                              ['Silhouette', 'Calinski-Harabasz', 'Davies-Bouldin']):
    ax.bar(ks_a, vals, color=colors_a, edgecolor='white', linewidth=0.8)
    if SELECTED_KA is not None:
        ax.axvline(SELECTED_KA, color='#d73027', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xlabel('k', fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_xticks(ks_a)
    ax.spines[['top', 'right']].set_visible(False)

_jac_a_s = str(round(JACCARD_KA_MEAN, 3)) if JACCARD_KA_MEAN is not None else 'N/A'
_sel_a_s = str(SELECTED_KA) if SELECTED_KA is not None else 'none'
fig.suptitle('Profile A Stage 2 k-selection  (selected k=' + _sel_a_s +
             ', Jaccard=' + _jac_a_s + ')', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'fig_f5_profileA_k_selection.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_f5_profileA_k_selection.png")


## 6. Assign combined cell labels

Profile B cells ordered by `sig_hat` ascending (B_behav1 = lowest noise).
Profile A cells ordered by `sig_hat` ascending (A_behav1 = lowest noise), if Stage 2 stable.
Combined label stored in `cell_f`.


In [ ]:
# ── Profile B final clustering ────────────────────────────────────────────────
km_B_final = KMeans(n_clusters=SELECTED_KB, init='k-means++', n_init=10, random_state=RANDOM_STATE)
raw_B = km_B_final.fit_predict(X_B)

B_sig_order = [(c, B_df['sig_hat'].values[raw_B == c].mean()) for c in range(SELECTED_KB)]
B_sig_order.sort(key=lambda x: x[1])
B_label_map = {orig: new for new, (orig, _) in enumerate(B_sig_order)}
B_labels = np.array([B_label_map[l] for l in raw_B])
B_cell_names = {i: 'B_behav' + str(i+1) for i in range(SELECTED_KB)}
B_df = B_df.copy()
B_df['stage2_B'] = [B_cell_names[l] for l in B_labels]

# ── Profile A final clustering ────────────────────────────────────────────────
if PROFILE_A_SPLIT and SELECTED_KA is not None:
    km_A_final = KMeans(n_clusters=SELECTED_KA, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    raw_A = km_A_final.fit_predict(X_A)
    A_sig_order = [(c, A_df['sig_hat'].values[raw_A == c].mean()) for c in range(SELECTED_KA)]
    A_sig_order.sort(key=lambda x: x[1])
    A_label_map = {orig: new for new, (orig, _) in enumerate(A_sig_order)}
    A_labels = np.array([A_label_map[l] for l in raw_A])
    A_cell_names = {i: 'A_behav' + str(i+1) for i in range(SELECTED_KA)}
    A_df = A_df.copy()
    A_df['stage2_A'] = [A_cell_names[l] for l in A_labels]
else:
    A_df = A_df.copy()
    A_df['stage2_A'] = 'A_all'
    SELECTED_KA = 1
    A_cell_names = {0: 'A_all'}
    A_labels = np.zeros(len(A_df), dtype=int)

# ── Combine into master df ────────────────────────────────────────────────────
df = df.copy()
b_map = dict(zip(B_df['participant_id_canonical'], B_df['stage2_B']))
a_map = dict(zip(A_df['participant_id_canonical'], A_df['stage2_A']))
df['stage2_B'] = df['participant_id_canonical'].map(b_map)
df['stage2_A'] = df['participant_id_canonical'].map(a_map)
df['cell_f'] = df['stage2_B'].fillna('') + df['stage2_A'].fillna('')
df['cell_f'] = df['cell_f'].replace('', np.nan)

B_CELL_ORDER = ['B_behav' + str(i+1) for i in range(SELECTED_KB)]
A_CELL_ORDER = sorted(A_df['stage2_A'].unique().tolist())
ALL_CELLS = B_CELL_ORDER + A_CELL_ORDER

cell_counts = df['cell_f'].value_counts()
print("Final cell assignments:")
for c in ALL_CELLS:
    print(f"  {c}: N={cell_counts.get(c, 0)}")
print()
min_n = min(cell_counts.get(c, 0) for c in ALL_CELLS)
print(f"All cells N >= {MIN_CLUSTER}: {min_n >= MIN_CLUSTER}  (min N={min_n})")
print(f"Total taxonomy cells: {len(ALL_CELLS)}")


## 7. Cell profiles and overlays

For each cell: mean/median of behavioral features, NeT overlay, condition independence,
and psychosocial Kruskal-Wallis. Condition tests are run within-profile (not full-sample)
to avoid confounding with the Stage 1 network split.


In [ ]:
PROFILE_FEATS_BEHAV = CORE_FEATURES + ['H_hour', 'mu_hat']
PROFILE_FEATS_NET   = ['HHI_net', 'rho_self', 'H_trans', 'H_app']

print("Cell profiles (behavioral features — raw units):")
print("-" * 80)
profile_rows = []
for cell in ALL_CELLS:
    sub = df[df['cell_f'] == cell]
    row = {'cell': cell, 'N': len(sub)}
    for feat in PROFILE_FEATS_BEHAV + PROFILE_FEATS_NET:
        valid = sub[feat].dropna()
        row[feat + '_mean']   = round(valid.mean(), 4)   if len(valid) > 0 else np.nan
        row[feat + '_median'] = round(valid.median(), 4) if len(valid) > 0 else np.nan
        row[feat + '_std']    = round(valid.std(), 4)    if len(valid) > 0 else np.nan
    profile_rows.append(row)

profiles_df = pd.DataFrame(profile_rows).set_index('cell')
show_cols = ['N'] + [f + '_mean' for f in CORE_FEATURES] + ['H_hour_mean'] + [f + '_mean' for f in PROFILE_FEATS_NET]
print(profiles_df[show_cols].to_string())


In [ ]:
print("Condition independence (χ² within each profile):")
print()
for profile_name, sub_df, stage_col, cell_order in [
    ('Profile B', B_df, 'stage2_B', B_CELL_ORDER),
    ('Profile A', A_df, 'stage2_A', A_CELL_ORDER)
]:
    ctab = pd.crosstab(sub_df[stage_col], sub_df['study_condition'])
    print(f"{profile_name}:")
    print(ctab)
    if ctab.shape[0] > 1 and ctab.shape[1] > 1:
        chi2, p, dof, _ = chi2_contingency(ctab)
        interp = "condition-independent" if p > 0.05 else "NOT condition-independent (caution)"
        print(f"  chi2={chi2:.3f}, df={dof}, p={p:.4f} — {interp}")
    else:
        print("  (insufficient groups for chi-square)")
    print()


In [ ]:
from scipy.stats import kruskal

print("Psychosocial Kruskal-Wallis (Bonferroni α=0.05/" + str(len(PSYCH_SCALES)) + "=0.010)")
N_SCALES = len(PSYCH_SCALES)

for profile_name, sub_df, stage_col, cell_order in [
    ('Profile B', B_df, 'stage2_B', B_CELL_ORDER),
    ('Profile A', A_df, 'stage2_A', A_CELL_ORDER)
]:
    print()
    print(f"{profile_name}:")
    n_sig = 0
    for scale in PSYCH_SCALES:
        groups = [sub_df[sub_df[stage_col] == c][scale].dropna().values for c in cell_order]
        groups = [g for g in groups if len(g) >= 3]
        if len(groups) < 2:
            print(f"  {scale:35s}: insufficient groups")
            continue
        try:
            H, p = kruskal(*groups)
            p_bonf = min(p * N_SCALES, 1.0)
            sig_flag = " *** SIGNIFICANT" if p_bonf < 0.05 else ""
            if p_bonf < 0.05:
                n_sig += 1
            print(f"  {scale:35s}: H={H:.3f}, p={p:.4f}, p_bonf={p_bonf:.4f}{sig_flag}")
        except Exception as e:
            print(f"  {scale:35s}: error — {e}")
    print(f"  Significant scales (Bonferroni): {n_sig}/{N_SCALES}")


## 8. Cross-path comparisons

Six comparisons quantify how the hierarchical Path F taxonomy relates to Phase 1,
Path C, and Path A. These are the core analytical deliverables of this notebook.


In [ ]:
from sklearn.metrics import adjusted_rand_score

print("=" * 65)
print("Comparison 1 — Stage 1 vs Phase 1 profile (validation check)")
print("=" * 65)
mask_s1 = df['cell_f'].notna() & df['profile_p1'].notna()
stage1_is_B = df.loc[mask_s1, 'cell_f'].str.startswith('B').astype(int)
p1_is_B     = (df.loc[mask_s1, 'profile_p1'] == 'Profile B').astype(int)
ari_s1_p1 = adjusted_rand_score(p1_is_B, stage1_is_B)
print(f"  ARI(Stage 1 vs Phase 1) = {ari_s1_p1:.4f}  (expected: 1.000)")
print()

print("=" * 65)
print("Comparison 2 — Stage 2 within Profile B vs Path A (within B)")
print("=" * 65)
mask2 = B_df['cluster_behav'].notna() & B_df['stage2_B'].notna()
ari_b2_pa = adjusted_rand_score(B_df.loc[mask2, 'cluster_behav'],
                                  B_df.loc[mask2, 'stage2_B'])
print(f"  N = {mask2.sum()}")
print(f"  ARI(Stage2-B vs Path A within B) = {ari_b2_pa:.4f}")
_interp2 = ("high correspondence — within-B structure mirrors full-sample Path A" if ari_b2_pa > 0.70
            else ("moderate correspondence — partial alignment with Path A" if ari_b2_pa > 0.40
                  else "low correspondence — within-B structure diverges from full-sample Path A"))
print(f"  Interpretation: {_interp2}")
print()

print("=" * 65)
print("Comparison 3 — Stage 2 within Profile A vs Path A (within A)")
print("=" * 65)
mask3 = A_df['cluster_behav'].notna() & A_df['stage2_A'].notna()
ari_a2_pa = adjusted_rand_score(A_df.loc[mask3, 'cluster_behav'],
                                  A_df.loc[mask3, 'stage2_A'])
print(f"  N = {mask3.sum()}")
print(f"  ARI(Stage2-A vs Path A within A) = {ari_a2_pa:.4f}")
_interp3 = ("high correspondence" if ari_a2_pa > 0.70
            else ("moderate correspondence" if ari_a2_pa > 0.40
                  else "low correspondence — within-A structure diverges from full-sample Path A"))
print(f"  Interpretation: {_interp3}")


In [ ]:
print("=" * 65)
print("Comparison 4 — Full Path F taxonomy vs Path A (C1/C2/C3)")
print("=" * 65)
mask4 = df['cell_f'].notna() & df['cluster_behav'].notna()
ari_f_a = adjusted_rand_score(df.loc[mask4, 'cell_f'],
                                df.loc[mask4, 'cluster_behav'])
print(f"  N = {mask4.sum()}")
print(f"  ARI(Path F vs Path A) = {ari_f_a:.4f}")
_interp4 = ("substantial overlap with Path A" if ari_f_a > 0.60
            else ("moderate overlap — Stage 1 NeT split introduces new information beyond Path A"
                  if ari_f_a > 0.30 else "low overlap — Path F taxonomy is distinct from Path A"))
print(f"  Interpretation: {_interp4}")
print()

print("=" * 65)
print("Comparison 5 — Full Path F taxonomy vs Path C (B1/B2/A1/A2)")
print("=" * 65)
mask5 = df['cell_f'].notna() & df['subprofile_C'].notna()
ari_f_c = adjusted_rand_score(df.loc[mask5, 'cell_f'],
                                df.loc[mask5, 'subprofile_C'])
print(f"  N = {mask5.sum()}")
print(f"  ARI(Path F vs Path C) = {ari_f_c:.4f}")
_interp5 = ("Path F substantially recovers Path C structure" if ari_f_c > 0.60
            else ("moderate overlap — Stage 2 behavioral split adds information beyond Path C"
                  if ari_f_c > 0.30 else "low overlap — Path F taxonomy is distinct from Path C"))
print(f"  Interpretation: {_interp5}")
print()

print("=" * 65)
print("Comparison 6 — Contingency: Path F cells × Path C sub-profiles")
print("=" * 65)
mask6 = df['cell_f'].notna() & df['subprofile_C'].notna()
ctab6 = pd.crosstab(df.loc[mask6, 'cell_f'], df.loc[mask6, 'subprofile_C'])
print("Count table:")
print(ctab6)
print()
print("Row % (within Path F cell):")
print((ctab6.div(ctab6.sum(axis=1), axis=0) * 100).round(1))

chi2_6, p6, dof6, _ = chi2_contingency(ctab6)
n6 = ctab6.values.sum()
cramers_v = float(np.sqrt(chi2_6 / (n6 * (min(ctab6.shape) - 1))))
print(f"\nChi-square: chi2={chi2_6:.3f}, df={dof6}, p={p6:.4f}")
print(f"Cramér's V = {cramers_v:.3f}")
_interp6 = ("strong association — network and behavioral sub-types are NOT fully independent" if cramers_v > 0.40
            else ("moderate association" if cramers_v > 0.20
                  else "weak association — network and behavioral sub-types are approximately independent"))
print(f"Interpretation: {_interp6}")
print()

print("Contingency: Path F cells × Path A clusters")
ctab_fa = pd.crosstab(df.loc[mask4, 'cell_f'], df.loc[mask4, 'cluster_behav'])
print("Count table:")
print(ctab_fa)
print()
print("Row % (within Path F cell):")
print((ctab_fa.div(ctab_fa.sum(axis=1), axis=0) * 100).round(1))


In [ ]:
print("Sensitivity check: within-group scaler vs full-sample scaler")
print()
# Re-run Stage 2 with full-sample scaler
km_B_sens = KMeans(n_clusters=SELECTED_KB, init='k-means++', n_init=10, random_state=RANDOM_STATE)
raw_B_sens = km_B_sens.fit_predict(X_B_full)
ari_sens_B = adjusted_rand_score(B_labels, raw_B_sens)
print(f"  Profile B: ARI(within-group vs full-sample scaler) = {ari_sens_B:.4f}")

if PROFILE_A_SPLIT and SELECTED_KA > 1:
    km_A_sens = KMeans(n_clusters=SELECTED_KA, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    raw_A_sens = km_A_sens.fit_predict(X_A_full)
    ari_sens_A = adjusted_rand_score(A_labels, raw_A_sens)
    print(f"  Profile A: ARI(within-group vs full-sample scaler) = {ari_sens_A:.4f}")
else:
    ari_sens_A = None
    print("  Profile A: sensitivity check not applicable (no Stage 2 split)")
print()
_sens_interp_b = ("scaling choice has negligible impact" if ari_sens_B > 0.90
                  else ("minor impact" if ari_sens_B > 0.70 else "scaling choice meaningfully alters solution"))
print(f"  Profile B interpretation: {_sens_interp_b}")


## 9. Figures


In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.axis('off')
fig.patch.set_facecolor('#fafafa')

def box(ax, x, y, w, h, text, fc='#e8f4f8', ec='#2c7bb6', fontsize=9.5):
    rect = mpatches.FancyBboxPatch((x - w/2, y - h/2), w, h,
                                    boxstyle='round,pad=0.08',
                                    facecolor=fc, edgecolor=ec, linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=fontsize,
            fontweight='bold', wrap=True, multialignment='center')

def arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#555555', lw=1.5))

# Root
box(ax, 5.0, 7.2, 3.2, 0.9,
    'All participants  N=414 (StoP-eligible, complete features)', fc='#f0f0f0', ec='#555555', fontsize=9)

# Stage 1 label
ax.text(5.0, 6.55, 'Stage 1: Phase 1 NeT-driven split  (profile_p1)', ha='center', fontsize=8.5,
        color='#555555', style='italic')

# Profile B
arrow(ax, 5.0, 6.75, 2.8, 5.9)
_nb = str(len(B_df))
_kb = str(SELECTED_KB)
_jb = str(round(JACCARD_KB_MEAN, 3)) if JACCARD_KB_MEAN is not None else 'N/A'
box(ax, 2.8, 5.5, 3.2, 0.9,
    'Profile B (hub-dominated)\nN=' + _nb + '  (HHI≈1.0, rho≈1.0)', fc='#dbe9f4', ec='#2c7bb6', fontsize=9)

# Profile A
arrow(ax, 5.0, 6.75, 7.2, 5.9)
_na = str(len(A_df))
box(ax, 7.2, 5.5, 3.2, 0.9,
    'Profile A (distributed)\nN=' + _na + '  (HHI≈0.58, rho≈0.88)', fc='#fde8d0', ec='#e08214', fontsize=9)

# Stage 2 labels
ax.text(2.8, 4.85, 'Stage 2: behavioral k-means  k=' + _kb + '\n(sig_hat, H_emp, x_mean; within-group scale)',
        ha='center', fontsize=8, color='#555555', style='italic')
ax.text(7.2, 4.85, 'Stage 2: behavioral k-means  k=' + _kb + '\n(sig_hat, H_emp, x_mean; within-group scale)',
        ha='center', fontsize=8, color='#555555', style='italic')

# Bugfix: adjacent leaf-cell boxes used to collide/overlap (fixed box width=2.6 vs.
# spacing that shrank whenever there were 3+ B cells, and a near-zero gap between the
# last B cell and the first A cell -- e.g. B_behav3/A_all visually merged into one
# run-on label). Fix: narrower boxes (2.6 -> 1.6), smaller leaf font (8.5 -> 7.5), and
# each group's cells are now spaced by a fixed pitch centered on that group's own
# parent box (2.8 for B, 7.2 for A) instead of a linspace across a fixed, too-narrow
# span -- this keeps a real gap both within a group and between the B/A groups
# regardless of how many cells SELECTED_KB / A_CELL_ORDER end up being.
LEAF_W = 1.6
LEAF_H = 0.75
LEAF_FONTSIZE = 7.5
LEAF_PITCH = 1.9  # > LEAF_W, guarantees a visible gap between adjacent leaf boxes

# B cells
B_COLORS_DIAG = ['#2c7bb6', '#74add1', '#abd9e9'][:SELECTED_KB]
b_offsets = np.arange(SELECTED_KB) - (SELECTED_KB - 1) / 2
b_xs = 2.8 + b_offsets * LEAF_PITCH
for i, (bx, cell) in enumerate(zip(b_xs, B_CELL_ORDER)):
    arrow(ax, 2.8, 5.05, bx, 3.7)
    n_cell = int(df['cell_f'].value_counts().get(cell, 0))
    sub = df[df['cell_f'] == cell]
    _sig = str(round(sub['sig_hat'].mean(), 3)) if len(sub) > 0 else '?'
    _xm  = str(round(sub['x_mean'].mean(), 3))  if len(sub) > 0 else '?'
    box(ax, bx, 3.3, LEAF_W, LEAF_H,
        cell + '  N=' + str(n_cell) + '\nsig=' + _sig + '  x=' + _xm,
        fc='#dbe9f4', ec='#2c7bb6', fontsize=LEAF_FONTSIZE)

# A cells
A_COLORS_DIAG = ['#e08214', '#fdb863'][:len(A_CELL_ORDER)]
a_offsets = np.arange(len(A_CELL_ORDER)) - (len(A_CELL_ORDER) - 1) / 2
a_xs = 7.2 + a_offsets * LEAF_PITCH
for i, (ax_x, cell) in enumerate(zip(a_xs, A_CELL_ORDER)):
    arrow(ax, 7.2, 5.05, ax_x, 3.7)
    n_cell = int(df['cell_f'].value_counts().get(cell, 0))
    sub = df[df['cell_f'] == cell]
    _sig = str(round(sub['sig_hat'].mean(), 3)) if len(sub) > 0 else '?'
    _xm  = str(round(sub['x_mean'].mean(), 3))  if len(sub) > 0 else '?'
    box(ax, ax_x, 3.3, LEAF_W, LEAF_H,
        cell + '  N=' + str(n_cell) + '\nsig=' + _sig + '  x=' + _xm,
        fc='#fde8d0', ec='#e08214', fontsize=LEAF_FONTSIZE)

# Summary stats
total_cells = len(ALL_CELLS)
ax.text(5.0, 2.4,
        'Total cells: ' + str(total_cells) + '     Stage 2 Jaccard (Profile B): ' + _jb + '  [' + JACCARD_KB_LABEL + ']'
        + '     Stage 1 validation ARI ≈ 1.000',
        ha='center', fontsize=9, color='#333333',
        bbox=dict(boxstyle='round', facecolor='#f5f5f5', edgecolor='#cccccc', alpha=0.8))

ax.text(5.0, 1.7,
        'Step 2 only — exploratory, DS5-specific. No causal claims. All labels require replication.',
        ha='center', fontsize=8.5, color='#666666', style='italic')

ax.set_title('Path F — Two-Stage Hierarchical Phenotype Taxonomy', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(FIGURES / 'fig_f1_hierarchy_structure.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_f1_hierarchy_structure.png")


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

x = np.arange(len(ALL_CELLS))
width = 0.22
FEAT_COLORS = {'sig_hat': '#d73027', 'H_emp': '#4575b4', 'x_mean': '#1a9641'}

for i, feat in enumerate(CORE_FEATURES):
    means = [profiles_df.loc[c, feat + '_mean'] for c in ALL_CELLS]
    stds  = [profiles_df.loc[c, feat + '_std']  for c in ALL_CELLS]
    offset = (i - 1) * width
    ax.bar(x + offset, means, width, yerr=stds, capsize=4,
           color=FEAT_COLORS[feat], alpha=0.85, label=feat,
           error_kw={'elinewidth': 1.2, 'ecolor': '#555555'})

ax.set_xticks(x)
ax.set_xticklabels(ALL_CELLS, fontsize=10)
ax.set_ylabel('Feature mean (raw units)', fontsize=10)
ax.set_title('Path F — Cell profiles: behavioral features\n(bars = mean ± 1 SD per cell)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')
ax.spines[['top', 'right']].set_visible(False)

# Annotate N per cell
for i, cell in enumerate(ALL_CELLS):
    n = int(df['cell_f'].value_counts().get(cell, 0))
    ax.text(i, -0.04, 'N=' + str(n), ha='center', va='top', fontsize=8.5,
            transform=ax.get_xaxis_transform())

# Vertical separator between B and A cells
ax.axvline(len(B_CELL_ORDER) - 0.5, color='#aaaaaa', linestyle='--', linewidth=1)
ax.text(len(B_CELL_ORDER) - 0.5 + 0.05, ax.get_ylim()[1] * 0.97,
        'Profile A →', fontsize=8, color='#666666')
ax.text(len(B_CELL_ORDER) - 0.5 - 0.05, ax.get_ylim()[1] * 0.97,
        '← Profile B', fontsize=8, color='#666666', ha='right')

plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig(FIGURES / 'fig_f6_cell_profiles.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_f6_cell_profiles.png")


In [ ]:
def draw_alluvial(ax, left_cats, right_cats, crosstab_df, left_colors, title=''):
    """Draw alluvial / flow diagram between two categorical variables."""
    total = crosstab_df.values.sum()
    gap_frac = 0.03

    # Compute bar heights and positions
    def compute_positions(cats, get_height):
        positions = {}
        y = 0.0
        for cat in cats:
            h = get_height(cat)
            positions[cat] = (y, y + h)
            y += h + gap_frac * total
        return positions

    left_pos  = compute_positions(left_cats,  lambda c: float(crosstab_df.loc[c].sum()))
    right_pos = compute_positions(right_cats, lambda c: float(crosstab_df[c].sum()))

    # Draw left bars
    for i, cat in enumerate(left_cats):
        y0, y1 = left_pos[cat]
        ax.fill_betweenx([y0, y1], [0]*2, [0.25]*2,
                          color=left_colors.get(cat, '#aaaaaa'), alpha=0.90)
        ax.text(-0.02, (y0+y1)/2, cat, ha='right', va='center', fontsize=8.5, fontweight='bold')

    # Draw right bars
    CELL_COLORS = {'B_behav1': '#2c7bb6', 'B_behav2': '#74add1', 'B_behav3': '#abd9e9',
                   'A_behav1': '#e08214', 'A_behav2': '#fdb863', 'A_all': '#f5a623'}
    for cat in right_cats:
        y0, y1 = right_pos[cat]
        rc = CELL_COLORS.get(cat, '#cccccc')
        ax.fill_betweenx([y0, y1], [0.75]*2, [1.0]*2, color=rc, alpha=0.90)
        ax.text(1.02, (y0+y1)/2, cat, ha='left', va='center', fontsize=8.5, fontweight='bold')

    # Draw flows
    t = np.linspace(0, 1, 80)
    smooth = 3*t**2 - 2*t**3   # smoothstep

    left_flow_y  = {c: left_pos[c][0]  for c in left_cats}
    right_flow_y = {c: right_pos[c][0] for c in right_cats}

    for lc in left_cats:
        for rc in right_cats:
            count = float(crosstab_df.loc[lc, rc]) if (lc in crosstab_df.index and rc in crosstab_df.columns) else 0
            if count < 0.5:
                continue
            ly0 = left_flow_y[lc];  ly1 = ly0 + count
            ry0 = right_flow_y[rc]; ry1 = ry0 + count
            xu = 0.25 + (0.75 - 0.25) * t
            y_top = ly1 + (ry1 - ly1) * smooth
            y_bot = ly0 + (ry0 - ly0) * smooth
            xs = np.concatenate([xu, xu[::-1]])
            ys = np.concatenate([y_top, y_bot[::-1]])
            poly = Polygon(np.column_stack([xs, ys]), closed=True,
                           facecolor=left_colors.get(lc, '#aaaaaa'), alpha=0.35, edgecolor='none')
            ax.add_patch(poly)
            left_flow_y[lc]  = ly1
            right_flow_y[rc] = ry1

    max_y = max(v[1] for v in list(left_pos.values()) + list(right_pos.values()))
    ax.set_ylim(-0.05 * total, max_y + 0.08 * total)
    ax.set_xlim(-0.35, 1.35)
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=10, fontweight='bold')

# Build cross-tab for alluvial
mask_al = df['cell_f'].notna() & df['cluster_behav'].notna()
ctab_al = pd.crosstab(df.loc[mask_al, 'cluster_behav'], df.loc[mask_al, 'cell_f'])

# Ensure all cells are present as columns
for c in ALL_CELLS:
    if c not in ctab_al.columns:
        ctab_al[c] = 0
ctab_al = ctab_al[ALL_CELLS]

fig, ax = plt.subplots(figsize=(9, 7))
draw_alluvial(ax,
              left_cats=['C1', 'C2', 'C3'],
              right_cats=ALL_CELLS,
              crosstab_df=ctab_al,
              left_colors=PA_COLORS,
              title='Path A clusters → Path F cells')

ax.text(0.125, -0.07 * ctab_al.values.sum(), 'Path A clusters',
        ha='center', fontsize=9, style='italic', transform=ax.transData)
ax.text(0.875, -0.07 * ctab_al.values.sum(), 'Path F cells',
        ha='center', fontsize=9, style='italic', transform=ax.transData)

fig.suptitle('Path A → Path F: how behavioral types distribute across\nhierarchical cells',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES / 'fig_f7_sankey_pathA_to_pathF.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_f7_sankey_pathA_to_pathF.png")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap helper
def heatmap(ax, ctab, title, cmap='Blues'):
    import matplotlib.colors as mcolors
    pct = (ctab.div(ctab.sum(axis=1), axis=0) * 100).round(1)
    im = ax.imshow(pct.values, cmap=cmap, aspect='auto', vmin=0, vmax=100)
    ax.set_xticks(range(len(ctab.columns)))
    ax.set_yticks(range(len(ctab.index)))
    ax.set_xticklabels(ctab.columns, rotation=30, ha='right', fontsize=9)
    ax.set_yticklabels(ctab.index, fontsize=9)
    ax.set_xlabel('Column category', fontsize=9)
    ax.set_ylabel('Row: Path F cell', fontsize=9)
    ax.set_title(title, fontsize=10, fontweight='bold')
    for i in range(len(ctab.index)):
        for j in range(len(ctab.columns)):
            n = int(ctab.values[i, j])
            p = float(pct.values[i, j])
            col = 'white' if p > 55 else 'black'
            ax.text(j, i, str(n) + '\n(' + str(p) + '%)', ha='center', va='center',
                    fontsize=8.5, color=col)
    plt.colorbar(im, ax=ax, label='Row %')

# Heatmap 1: Path F × Path C
mask_fc2 = df['cell_f'].notna() & df['subprofile_C'].notna()
ctab_h1 = pd.crosstab(df.loc[mask_fc2, 'cell_f'], df.loc[mask_fc2, 'subprofile_C'])
ctab_h1 = ctab_h1.reindex(ALL_CELLS).fillna(0).astype(int)
heatmap(axes[0], ctab_h1, 'Path F cells × Path C sub-profiles\n(rows = Path F cells)', cmap='Blues')

# Heatmap 2: Path F × Path A
mask_fa2 = df['cell_f'].notna() & df['cluster_behav'].notna()
ctab_h2 = pd.crosstab(df.loc[mask_fa2, 'cell_f'], df.loc[mask_fa2, 'cluster_behav'])
ctab_h2 = ctab_h2.reindex(ALL_CELLS).fillna(0).astype(int)
heatmap(axes[1], ctab_h2, 'Path F cells × Path A clusters\n(rows = Path F cells)', cmap='Oranges')

plt.tight_layout()
plt.savefig(FIGURES / 'fig_f8_contingency_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_f8_contingency_heatmaps.png")


## 10. Save outputs

All outputs saved to `outputs/model_outputs/phenotyping/`.
Companion `.md` files document unit of observation, column reference, and known limitations.


In [ ]:
# ── ds5_hierarchical_assignments.csv ─────────────────────────────────────────
assignments = df[['participant_id_canonical', 'study_condition', 'profile_p1',
                   'stage2_B', 'stage2_A', 'cell_f', 'cluster_behav', 'subprofile_C']].copy()
assignments.to_csv(PHENOTYPING / 'ds5_hierarchical_assignments.csv', index=False)
print(f"Saved ds5_hierarchical_assignments.csv  (N={len(assignments)})")
print(assignments['cell_f'].value_counts().reindex(ALL_CELLS).to_string())

# ── Companion .md ─────────────────────────────────────────────────────────────
_n_cells_s = str(len(ALL_CELLS))
_cells_s   = ', '.join(ALL_CELLS)
_nb_s      = str(len(B_df))
_na_s      = str(len(A_df))
_sel_kb_s  = str(SELECTED_KB)
_sel_ka_s  = str(SELECTED_KA)
_jb_s      = str(round(JACCARD_KB_MEAN, 3)) if JACCARD_KB_MEAN is not None else 'N/A'
_jka_s     = str(round(JACCARD_KA_MEAN, 3)) if JACCARD_KA_MEAN is not None else 'N/A'

companion_assign = (
    "# ds5_hierarchical_assignments.csv — Column Reference\n\n"
    "**Produced by:** `103_ds5_hierarchical_phenotype.ipynb`  \n"
    "**Date:** 2026-04-19  \n"
    "**Unit of observation:** One row per StoP-eligible participant (N=414)  \n\n"
    "## What this file contains\n\n"
    "Per-participant two-stage hierarchical phenotype assignments. "
    "Stage 1 = Phase 1 network-structural split. "
    "Stage 2 = behavioral k-means within each profile. "
    "Final combined label: `cell_f`.\n\n"
    "## Column reference\n\n"
    "| Column | Type | Description |\n"
    "|---|---|---|\n"
    "| `participant_id_canonical` | str | Unique participant identifier |\n"
    "| `study_condition` | str | Friction condition (control/default/reintervention) |\n"
    "| `profile_p1` | str | Phase 1 (Stage 1) split: Profile A or Profile B |\n"
    "| `stage2_B` | str or NaN | Stage 2 behavioral label within Profile B; NaN for Profile A participants |\n"
    "| `stage2_A` | str or NaN | Stage 2 behavioral label within Profile A; NaN for Profile B participants |\n"
    "| `cell_f` | str | Combined hierarchical cell label (" + _cells_s + ") |\n"
    "| `cluster_behav` | str | Path A behavior-only cluster label (C1/C2/C3; post-hoc comparison) |\n"
    "| `subprofile_C` | str | Path C network sub-cluster label (B1/B2/A1/A2; post-hoc comparison) |\n\n"
    "## Taxonomy details\n\n"
    "- Profile B (N=" + _nb_s + "): Stage 2 selected k=" + _sel_kb_s + "; Jaccard=" + _jb_s + " [" + JACCARD_KB_LABEL + "]\n"
    "- Profile A (N=" + _na_s + "): Stage 2 selected k=" + _sel_ka_s + "; Jaccard=" + _jka_s + " [" + JACCARD_KA_LABEL + "]\n"
    "- Total cells: " + _n_cells_s + "\n\n"
    "## How to load\n\n"
    "```python\n"
    "import pandas as pd\n"
    "df = pd.read_csv('outputs/model_outputs/phenotyping/ds5_hierarchical_assignments.csv')\n"
    "```\n\n"
    "## Known limitations\n\n"
    "1. Stage 2 is exploratory; cluster labels are DS5-specific and require cross-dataset replication.\n"
    "2. Condition independence is not guaranteed — Path A found condition chi2=91.885, p<0.0001 for full-sample behavioral clusters.\n"
    "3. Profile A Stage 2 has lower N (125) than Profile B (289) — stability claims for Profile A are weaker.\n"
    "4. No causal interpretation — network type does not cause behavioral type.\n"
)
with open(PHENOTYPING / 'ds5_hierarchical_assignments.md', 'w') as f:
    f.write(companion_assign)
print("Saved ds5_hierarchical_assignments.md")


In [ ]:
# ── ds5_hierarchical_profiles.csv ────────────────────────────────────────────
profiles_df.reset_index().to_csv(PHENOTYPING / 'ds5_hierarchical_profiles.csv', index=False)
print(f"Saved ds5_hierarchical_profiles.csv  ({len(profiles_df)} rows x {len(profiles_df.columns)+1} cols)")

companion_profiles = (
    "# ds5_hierarchical_profiles.csv — Column Reference\n\n"
    "**Produced by:** `103_ds5_hierarchical_phenotype.ipynb`  \n"
    "**Date:** 2026-04-19  \n"
    "**Unit of observation:** One row per Path F cell (" + str(len(ALL_CELLS)) + " rows)  \n\n"
    "## What this file contains\n\n"
    "Per-cell descriptive statistics (mean, median, std) for behavioral features "
    "(sig_hat, H_emp, x_mean, H_hour, mu_hat) and NeT overlay features "
    "(HHI_net, rho_self, H_trans, H_app). All values in raw (unscaled) units.\n\n"
    "## Column reference\n\n"
    "| Column | Description |\n"
    "|---|---|\n"
    "| `cell` | Path F cell label (e.g., B_behav1, A_behav2) |\n"
    "| `N` | Participants in cell |\n"
    "| `<feature>_mean` | Mean of feature in raw units |\n"
    "| `<feature>_median` | Median of feature in raw units |\n"
    "| `<feature>_std` | Standard deviation of feature in raw units |\n\n"
    "## Notes\n\n"
    "- All features reported in raw (unscaled) units for interpretability.\n"
    "- Stage 2 clustering used within-group StandardScaler internally; profiles are in original scale.\n"
    "- mu_hat and sigma_mu_ratio are near-zero/NaN for many DS5 participants; interpret with caution.\n\n"
    "## How to load\n\n"
    "```python\n"
    "import pandas as pd\n"
    "df = pd.read_csv('outputs/model_outputs/phenotyping/ds5_hierarchical_profiles.csv')\n"
    "```\n"
)
with open(PHENOTYPING / 'ds5_hierarchical_profiles.md', 'w') as f:
    f.write(companion_profiles)
print("Saved ds5_hierarchical_profiles.md")


In [ ]:
# ── ds5_hierarchical_stability.csv ───────────────────────────────────────────
stab_rows = []
for r in b_results:
    row = {'profile': 'Profile B', 'k': r['k'], 'valid': r['valid'],
           'silhouette': r['silhouette'], 'CH': r['CH'], 'DB': r['DB'],
           'min_cell': r['min_cell'], 'scaler': 'within-group'}
    if r['k'] in jaccard_B:
        row['jaccard_mean'] = jaccard_B[r['k']]['mean']
        row['jaccard_std']  = jaccard_B[r['k']]['std']
        row['jaccard_label'] = jaccard_B[r['k']]['label']
    else:
        row['jaccard_mean'] = None
        row['jaccard_std']  = None
        row['jaccard_label'] = None
    stab_rows.append(row)

for r in a_results:
    row = {'profile': 'Profile A', 'k': r['k'], 'valid': r['valid'],
           'silhouette': r['silhouette'], 'CH': r['CH'], 'DB': r['DB'],
           'min_cell': r['min_cell'], 'scaler': 'within-group'}
    if r['valid'] and r['k'] == SELECTED_KA and JACCARD_KA_MEAN is not None:
        row['jaccard_mean']  = JACCARD_KA_MEAN
        row['jaccard_std']   = JACCARD_KA_STD
        row['jaccard_label'] = JACCARD_KA_LABEL
    else:
        row['jaccard_mean']  = None
        row['jaccard_std']   = None
        row['jaccard_label'] = None
    stab_rows.append(row)

stab_df = pd.DataFrame(stab_rows)
stab_df.to_csv(PHENOTYPING / 'ds5_hierarchical_stability.csv', index=False)
print("Saved ds5_hierarchical_stability.csv")
print(stab_df[['profile', 'k', 'valid', 'silhouette', 'min_cell', 'jaccard_mean', 'jaccard_label']].to_string(index=False))


In [ ]:
# ── model_run_summary_path_f.md ──────────────────────────────────────────────

_n_total = str(len(df))
_nb2 = str(len(B_df)); _na2 = str(len(A_df))
_sel_kb2 = str(SELECTED_KB); _sel_ka2 = str(SELECTED_KA)
_jb2 = str(round(JACCARD_KB_MEAN, 3)) if JACCARD_KB_MEAN is not None else 'N/A'
_jka2 = str(round(JACCARD_KA_MEAN, 3)) if JACCARD_KA_MEAN is not None else 'N/A'
_jkb_std = str(round(JACCARD_KB_STD, 3)) if JACCARD_KB_STD is not None else 'N/A'
_jka_std = str(round(JACCARD_KA_STD, 3)) if JACCARD_KA_STD is not None else 'N/A'
_cells_s2 = ', '.join(ALL_CELLS)
_n_cells2 = str(len(ALL_CELLS))
_ari_s1p1 = str(round(ari_s1_p1, 4))
_ari_b2pa = str(round(ari_b2_pa, 4))
_ari_a2pa = str(round(ari_a2_pa, 4))
_ari_fa   = str(round(ari_f_a,   4))
_ari_fc   = str(round(ari_f_c,   4))
_cramersv = str(round(cramers_v,  3))
_chi26    = str(round(chi2_6, 3))
_p6       = str(round(p6, 4))

cell_profile_table_rows = []
for cell in ALL_CELLS:
    sub = df[df['cell_f'] == cell]
    _n  = str(len(sub))
    _sig = str(round(sub['sig_hat'].mean(), 4)) if len(sub) > 0 else 'N/A'
    _hep = str(round(sub['H_emp'].mean(), 4))   if len(sub) > 0 else 'N/A'
    _xm  = str(round(sub['x_mean'].mean(), 4))  if len(sub) > 0 else 'N/A'
    cell_profile_table_rows.append("| " + cell + " | " + _n + " | " + _sig + " | " + _hep + " | " + _xm + " |")
_profile_table = "\n".join(cell_profile_table_rows)

summary = (
    "# DS5 Step 2 Path F — Hierarchical Phenotyping: Model Run Summary\n\n"
    "**Date:** 2026-04-19\n"
    "**Notebook:** 103_ds5_hierarchical_phenotype.ipynb\n"
    "**Stage:** Stage 4, Step 2 — Path F\n\n"
    "---\n\n"
    "## Run parameters\n"
    "- Core Stage 2 features: " + str(CORE_FEATURES) + "\n"
    "- Stage 1 split: profile_p1 (Phase 1 assignments)\n"
    "- Within-group StandardScaler (primary)\n"
    "- RANDOM_STATE: 42\n"
    "- Bootstrap: 100 iterations, 80% sample fraction\n"
    "- Min cluster size threshold: 30\n"
    "- Analysis population: N = " + _n_total + "\n\n"
    "## Stage 2 k-selection results\n\n"
    "### Profile B (N=" + _nb2 + ")\n"
    "- Selected k: " + _sel_kb2 + "\n"
    "- Bootstrap Jaccard: " + _jb2 + " ± " + _jkb_std + " [" + JACCARD_KB_LABEL + "]\n\n"
    "### Profile A (N=" + _na2 + ")\n"
    "- Selected k: " + _sel_ka2 + "\n"
    "- Profile A Stage 2 split: " + str(PROFILE_A_SPLIT) + "\n"
    "- Bootstrap Jaccard: " + _jka2 + " ± " + _jka_std + " [" + JACCARD_KA_LABEL + "]\n\n"
    "## Final taxonomy\n\n"
    "- Total cells: " + _n_cells2 + "\n"
    "- Cell labels: " + _cells_s2 + "\n\n"
    "## Cell profiles (mean values)\n\n"
    "| Cell | N | sig_hat | H_emp | x_mean |\n"
    "|---|---|---|---|---|\n"
    + _profile_table + "\n\n"
    "## Cross-path comparisons\n\n"
    "- ARI(Stage 1 vs Phase 1):               " + _ari_s1p1 + "  (expected 1.000)\n"
    "- ARI(Stage2-B vs Path A within B):      " + _ari_b2pa + "\n"
    "- ARI(Stage2-A vs Path A within A):      " + _ari_a2pa + "\n"
    "- ARI(Path F combined vs Path A):        " + _ari_fa + "\n"
    "- ARI(Path F combined vs Path C):        " + _ari_fc + "\n"
    "- Cramér's V (Path F × Path C):          " + _cramersv + "\n"
    "- Chi-square (Path F × Path C sub):      " + _chi26 + ", p=" + _p6 + "\n\n"
    "## Output files\n"
    "- ds5_hierarchical_assignments.csv\n"
    "- ds5_hierarchical_assignments.md\n"
    "- ds5_hierarchical_profiles.csv\n"
    "- ds5_hierarchical_profiles.md\n"
    "- ds5_hierarchical_stability.csv\n"
    "- figures/fig_f1 through fig_f8\n\n"
    "## Key limitations\n"
    "1. Stage 2 behavioral clusters are exploratory and DS5-specific\n"
    "2. Condition independence not guaranteed (Path A found chi2=91.885, p<0.0001 full-sample)\n"
    "3. Profile A Stage 2 has lower power (N=125) than Profile B (N=289)\n"
    "4. No causal interpretation of network vs behavioral type relationship\n"
    "5. Cross-dataset replication required before any claim of generalizability\n"
)

with open(PHENOTYPING / 'model_run_summary_path_f.md', 'w') as f:
    f.write(summary)
print("Saved model_run_summary_path_f.md")
print()
print("All outputs saved to:", str(PHENOTYPING))


In [ ]:
import os
expected_files = [
    'ds5_hierarchical_assignments.csv',
    'ds5_hierarchical_assignments.md',
    'ds5_hierarchical_profiles.csv',
    'ds5_hierarchical_profiles.md',
    'ds5_hierarchical_stability.csv',
    'model_run_summary_path_f.md',
]
expected_figs = [
    'fig_f1_hierarchy_structure.png',
    'fig_f2_profileB_behav_space.png',
    'fig_f3_profileA_behav_space.png',
    'fig_f4_profileB_k_selection.png',
    'fig_f5_profileA_k_selection.png',
    'fig_f6_cell_profiles.png',
    'fig_f7_sankey_pathA_to_pathF.png',
    'fig_f8_contingency_heatmaps.png',
]

print("Output files:")
for f in expected_files:
    path = PHENOTYPING / f
    exists = path.exists()
    size = os.path.getsize(path) if exists else 0
    print(f"  {'OK' if exists else 'MISSING':7s} {f}  ({size} bytes)")

print()
print("Figures:")
for f in expected_figs:
    path = FIGURES / f
    exists = path.exists()
    size = os.path.getsize(path) if exists else 0
    print(f"  {'OK' if exists else 'MISSING':7s} {f}  ({size} bytes)")

print()
print("Path F taxonomy summary:")
print(df['cell_f'].value_counts().reindex(ALL_CELLS).to_string())
